In [ ]:
import time
import datetime
import random
import numpy as np
import polars as pl
from scipy.sparse import csr_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
import seaborn as sns

In [ ]:
books = pl.read_ipc('./data/goodreads_books.feather')
interactions = pl.read_ipc('./data/goodreads_interactions.feather')

In [ ]:
books.drop_in_place('isbn')
books.drop_in_place('series')
books.drop_in_place('popular_shelves')
books.drop_in_place('asin')
books.drop_in_place('kindle_asin')
books.drop_in_place('similar_books')
books.drop_in_place('description')
books.drop_in_place('link')
books.drop_in_place('authors')
books.drop_in_place('publisher')
books.drop_in_place('isbn13')
books.drop_in_place('edition_information')
books.drop_in_place('url')
books.drop_in_place('image_url')
books.drop_in_place('work_id')
books.drop_in_place('title')
books.drop_in_place('title_without_series')
books.drop_in_place('publication_month')
books.drop_in_place('publication_day')

In [ ]:
books = books.with_columns(pl.col('book_id').cast(pl.Int64))
books = books.filter(pl.col('publication_year').ne(''))
books = books.filter(pl.col('publication_year').cast(pl.Int64).le(2026))

In [ ]:
interactions = interactions.filter(pl.col('is_read').eq(1))
interactions = interactions.drop('is_read')
interactions = interactions.filter(pl.col('user_id').is_duplicated())

In [ ]:
books['text_reviews_count'].fill_null(0)

In [ ]:
books = books.with_columns(pl.col('text_reviews_count').str.replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('is_ebook').replace('false', '0').replace('true', '1').cast(pl.Float32))
books = books.with_columns(pl.col('average_rating').str.replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('num_pages').str.replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('publication_year').cast(pl.Float32))
books = books.with_columns(pl.col('ratings_count').replace('', '0').cast(pl.Float32))
books = books.with_columns(pl.col('country_code').replace('', '0').replace('US', '1').cast(pl.Float32))

In [ ]:
users = books.join(interactions, on='book_id')

In [ ]:
users = users.to_dummies('language_code')
users = users.to_dummies('format')

In [ ]:
def normalize_ids(df: pl.DataFrame, id: str):
    uniq_ids = set(df[id].unique())
    ids_to_indices = { orig_id : new_id for new_id, orig_id in enumerate(uniq_ids) }
    new_ids = df.with_columns(pl.col(id).replace(ids_to_indices).cast(pl.Int64))
    indices_to_ids = { new_id : orig_id for orig_id, new_id in ids_to_indices.items()}
    return new_ids, indices_to_ids

In [ ]:
users, _ = normalize_ids(users, 'user_id')

users_n_books = users['book_id'].n_unique()

In [ ]:
users.group_by('user_id').agg(pl.exclude('book_id').mean())

In [ ]:
gss = GroupShuffleSplit(n_splits=5)

users_train_idx, users_test_idx = next(gss.split(X=users, groups=users['user_id']))
users_train: pl.DataFrame = users[users_train_idx]
users_test:  pl.DataFrame = users[users_test_idx]

In [ ]:
users_test = users_test.filter(pl.col('user_id').is_duplicated())

In [ ]:
users_train, users_train_map_id = normalize_ids(users_train, 'user_id')
users_test,  users_test_map_id  = normalize_ids(users_test, 'user_id')

users_train_csr = csr_matrix((users_train['rating'], (users_train['user_id'], users_train['book_id'])), shape=(users_train['user_id'].n_unique(), users_n_books))

In [ ]:
users_test_seen,   users_test_unseen         = train_test_split(users_test, train_size=0.7, stratify=users_test['user_id'])

users_test_seen,   users_test_seen_map_id    = normalize_ids(users_test_seen,   'user_id')
users_test_unseen, users_test_unseen_map_id  = normalize_ids(users_test_unseen, 'user_id')

users_test_seen_csr = csr_matrix((users_test_seen['rating'], (users_test_seen['user_id'], users_test_seen['book_id'])), shape=(users_test_seen['user_id'].n_unique(), users_n_books))

In [ ]:
users_train = users_train.group_by('user_id')
users_test_seen = users_test_seen.group_by('user_id').agg(pl.exclude('book_id').mean())
users_test_unseen = users_test_unseen.group_by('user_id').agg(pl.exclude('book_id').mean())

In [ ]:
def create_kn(k, train, test, metric='cosine', n_jobs=24, return_distance=False):
    nn = NearestNeighbors(n_neighbors=k, metric=metric, n_jobs=n_jobs)
    nn.fit(train)
    return nn.kneighbors(test, return_distance=return_distance)

In [ ]:
def k_average_ratings(train_df: pl.DataFrame, test_seen_df: pl.DataFrame, test_unseen_df: pl.DataFrame, train_map_id, test_map_id, kn, k):
    kn_idx = kn[:, :k]
    nn = pl.DataFrame({ 'neighbor_idx' : kn_idx.reshape(-1), 'query_idx' : np.arange(test_seen_df['user_id'].n_unique()).repeat(k)})
    nb = nn.with_columns((pl.col('neighbor_idx').replace(train_map_id).alias('training_user_id'), pl.col('query_idx').replace(test_map_id).alias('query_user_id'))).select(['training_user_id', 'query_user_id'])
    tf = train_df.lazy().filter(pl.col('user_id').is_in(nb['training_user_id'].implode()))
    jn = tf.join(nb.lazy(), left_on='user_id', right_on='training_user_id').select('query_user_id', 'item_id', 'rating')
    pd = jn.group_by(['query_user_id', 'item_id'])
    pa = pd.agg(pl.col('rating').mean()).rename({ 'query_user_id': 'user_id', 'rating': 'pred_rating' })
    pf = pa.filter(pl.col('user_id').is_in(test_unseen_df['user_id'].implode()) & pl.col('item_id').is_in(test_unseen_df['item_id'].implode()))
    rt = pf.join(test_unseen_df.lazy(), on=['user_id', 'item_id']).filter(pl.col('rating').is_not_null()).collect(engine="streaming")
    pr = pearsonr(rt['pred_rating'].to_numpy(), rt['rating'].to_numpy())
    return pr.statistic ** 2

In [ ]:
users_kn = create_kn(200, users_train, users_test_seen)

In [ ]:
users_pr_5 = k_average_ratings(train_df=users_train,
                               test_seen_df=users_test_seen,
                               test_unseen_df=users_test_unseen,
                               train_map_id=users_train_map_id,
                               test_map_id=users_test_seen_map_id,
                               kn=users_kn,
                               k=5)
print(f'k=5 R^2: {users_pr_5}')